# Stage 1 — Non-Instruction Fine-Tuning

**Domain:** NovaDesk IT Helpdesk Assistant  
**Base model:** `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit`  
**Method:** QLoRA-style 4-bit loading with trainable LoRA adapters

This notebook performs continued domain adaptation on 66 raw knowledge-base paragraphs. It also records base-model answers to the ten fixed evaluation questions before training.

In [ ]:
# Run on a Google Colab GPU runtime (T4 or better).
!pip -q install -U unsloth trl transformers datasets peft accelerate bitsandbytes sentencepiece

In [ ]:
from pathlib import Path
import json, os, sys, torch

# Expected workflow: clone the GitHub repository, then open the notebook from the repo.
CANDIDATES = [Path.cwd(), Path('/content/domain-ai-assistant-finetuning')]
ROOT = next((p for p in CANDIDATES if (p / 'data').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Repository root not found. Clone the repo into /content/domain-ai-assistant-finetuning or run from the repo root.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Repository:', ROOT)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU runtime is required for these notebooks.')

In [ ]:
from datasets import Dataset
from peft import PeftModel
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported
from common import generate_answer, write_comparison_report

BASE_MODEL = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 1024
OUTPUT_DIR = ROOT / 'outputs/non_instruction_adapter'
ARTIFACT_DIR = ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)

## 1. Load and inspect raw domain text

In [ ]:
raw_text = (ROOT / 'data/non_instruction_data.txt').read_text(encoding='utf-8')
paragraphs = [p.strip() for p in raw_text.split('\n\n') if p.strip()]
print('Paragraphs:', len(paragraphs))
print(paragraphs[0])
assert len(paragraphs) >= 50

## 2. Clean and chunk text

In [ ]:
def clean_text(text: str) -> str:
    return ' '.join(text.replace('\u00a0', ' ').split())

cleaned = [clean_text(p) for p in paragraphs]
# Each paragraph is already short enough for the 1,024-token training context.
dataset = Dataset.from_dict({'text': cleaned}).train_test_split(test_size=0.1, seed=42)
print(dataset)

## 3. Load the 4-bit base model and test it before training

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

questions = json.loads((ROOT / 'data/evaluation_questions.json').read_text())
FastLanguageModel.for_inference(model)
base_answers = [generate_answer(model, tokenizer, q) for q in questions]
(ARTIFACT_DIR / 'base_outputs.json').write_text(json.dumps(base_answers, indent=2), encoding='utf-8')
write_comparison_report(
    ROOT / 'reports/base_model_evaluation.md',
    'Base Model Evaluation',
    questions,
    [
        ('Base model answer', base_answers),
        ('Problem observed', ['Complete after human review'] * len(questions)),
    ],
)
print('Updated reports/base_model_evaluation.md')
for q, a in zip(questions[:2], base_answers[:2]):
    print('Q:', q, '\nA:', a, '\n')

## 4. Apply LoRA adapters and train on raw text

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=SFTConfig(
        dataset_text_field='text',
        max_length=MAX_SEQ_LENGTH,
        packing=True,
        output_dir=str(ROOT / 'outputs/non_instruction_training'),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        logging_steps=1,
        eval_strategy='epoch',
        save_strategy='epoch',
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=42,
        report_to='none',
    ),
)
train_result = trainer.train()
print(train_result)
(ARTIFACT_DIR / 'non_instruction_train_metrics.json').write_text(
    json.dumps(train_result.metrics, indent=2, default=str), encoding='utf-8'
)
trainer.save_state()

## 5. Save and test the non-instruction adapter

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
FastLanguageModel.for_inference(model)
non_instruction_answers = [generate_answer(model, tokenizer, q) for q in questions]
(ARTIFACT_DIR / 'non_instruction_outputs.json').write_text(
    json.dumps(non_instruction_answers, indent=2), encoding='utf-8'
)
for q, answer in zip(questions[:3], non_instruction_answers[:3]):
    print('Q:', q)
    print('A:', answer, '\n')
print('Saved:', OUTPUT_DIR)

## Evidence to capture

Save a screenshot showing: GPU runtime, dataset size, trainable parameter count, training loss, and the final adapter path. Do not claim quality improvement from Stage 1 alone; the formal comparison is performed after SFT and DPO.